# FixMyCity — Dataset Cleanup + Retrain on Colab GPU

**CRITICAL**: The existing dataset has ~30% contamination (NSFW, anime, unrelated images scraped by a broken web scraper). This notebook:

1. **Cleans** the dataset — removes all contaminated file prefixes
2. **Downloads** real-world images from verified public datasets (RDD2022, TACO, flood datasets)
3. **Augments** with JPEG compression simulation to match frontend pipeline
4. **Trains** a 4-class EfficientNetV2S civic classifier
5. **Calibrates** temperature scaling thresholds
6. **Exports** TFJS model + artifacts for download

## Before you start
1. Runtime → Change runtime type → **T4 GPU** (or A100 on Pro) → Save
2. Upload to Google Drive folder `FixMyCity/`:
   - `my_dataset.zip` (your current dataset — will be cleaned automatically)
   - `train_civic_model.py`
   - `temperature_scaling.py`
   - `civic_labels.json`
3. Run all cells top to bottom (Shift+Enter)

### Contamination found (July 2026 audit)
| Prefix | Category | Count | Content |
|--------|----------|-------|---------|
| `drain_*` | drainage | 193 | NSFW anime, trains, posters |
| `scrape_*` | drainage | 447 | TV posters, puppies, anime |
| `bing_*` | drainage | 20 | Fashion photos |
| `scrape_*` | others | 639 | Solar panels, random web images |
| `oth_*` (all) | others | ~350 | Cricket, video games, zodiac, flutes |
| `kag_*` | others | 12 | Laptop batteries, misc products |

**Clean sources kept**: `nst_dr_image_*` (drainage), `kag_*`/`kg_potholes_*`/`nst_ph_*` (potholes), `kg_streetlight_*`/`nst_sl_*` (streetlight)

## 1. Verify GPU

In [ ]:
!nvidia-smi -L
# Expect a line like: GPU 0: Tesla T4 ...  (if empty -> set Runtime accelerator to T4 GPU)

## 2. Install pinned deps
Pin `tensorflow==2.15` + matching `tensorflowjs` so the TFJS **LayersModel** export matches what `server.js` loads. Restart is NOT needed.

In [ ]:
!pip -q install tensorflow==2.15.0 tensorflowjs==4.17.0 scikit-learn pillow 2>&1 | tail -5
import tensorflow as tf
print('TF', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))
print('Backbone:', 'EfficientNetV2S' if hasattr(tf.keras.applications, 'EfficientNetV2S') else 'EfficientNetB0 (fallback)')

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SRC = '/content/drive/MyDrive/FixMyCity'
print('files in Drive/FixMyCity:', os.listdir(SRC))

## 4. Stage files + unzip dataset

In [ ]:
import shutil, os, glob
os.makedirs('/content/work', exist_ok=True)
os.chdir('/content/work')
for f in ['train_civic_model.py', 'civic_labels.json']:
    shutil.copy(os.path.join(SRC, f), f)
if not os.path.isdir('my_dataset'):
    !unzip -q -o "$SRC/my_dataset.zip" -d /content/work

print("=== BEFORE cleanup ===")
for c in ['drainage','others','potholes','streetlight']:
    n = len(glob.glob(f'my_dataset/{c}/*'))
    print(f'  {c}: {n}')

## 4b. CLEANUP — Remove contaminated images

Removes all file prefixes confirmed as junk during the July 2026 audit. Moves them to `_quarantine/` (not deleted, in case you want to inspect).

In [ ]:
import os, glob, shutil

QUARANTINE = '/content/work/_quarantine'
os.makedirs(QUARANTINE, exist_ok=True)

# Contaminated prefixes per category (confirmed by visual inspection)
JUNK_RULES = {
    'drainage': ['drain_', 'scrape_', 'bing_'],
    'others':   ['scrape_', 'oth_', 'kag_'],  # entire others category is contaminated
}

total_removed = 0
for category, prefixes in JUNK_RULES.items():
    cat_dir = f'my_dataset/{category}'
    if not os.path.isdir(cat_dir):
        continue
    q_dir = os.path.join(QUARANTINE, category)
    os.makedirs(q_dir, exist_ok=True)
    removed = 0
    for f in os.listdir(cat_dir):
        if any(f.startswith(p) for p in prefixes):
            shutil.move(os.path.join(cat_dir, f), os.path.join(q_dir, f))
            removed += 1
    print(f'{category}: removed {removed} junk images → _quarantine/{category}/')
    total_removed += removed

# Also remove tiny files (<5KB = likely corrupt) and convert PNGs to JPG
for category in ['drainage', 'others', 'potholes', 'streetlight']:
    cat_dir = f'my_dataset/{category}'
    if not os.path.isdir(cat_dir):
        continue
    for f in os.listdir(cat_dir):
        fpath = os.path.join(cat_dir, f)
        # Remove tiny files
        if os.path.getsize(fpath) < 5000:
            q_dir = os.path.join(QUARANTINE, category)
            os.makedirs(q_dir, exist_ok=True)
            shutil.move(fpath, os.path.join(q_dir, f))
            total_removed += 1
            print(f'  {category}/{f}: removed (too small: {os.path.getsize(os.path.join(q_dir, f))} bytes)')
        # Convert PNG to JPG
        elif f.lower().endswith('.png'):
            try:
                from PIL import Image
                img = Image.open(fpath).convert('RGB')
                jpg_path = fpath.rsplit('.', 1)[0] + '.jpg'
                img.save(jpg_path, 'JPEG', quality=95)
                os.remove(fpath)
                print(f'  {category}/{f}: converted PNG → JPG')
            except Exception as e:
                print(f'  {category}/{f}: PNG convert failed ({e}), removing')
                os.remove(fpath)

print(f'\n=== Total removed: {total_removed} junk images ===')
print('\n=== AFTER cleanup ===')
for c in ['drainage','others','potholes','streetlight']:
    n = len(glob.glob(f'my_dataset/{c}/*'))
    print(f'  {c}: {n}')

## 4c. DOWNLOAD — Premium datasets from verified public sources

Downloads real-world, CC-licensed images to build a robust training set. **Target: 800–2000 images per category**.

| Source | Category | License | Images | Source Type |
|--------|----------|---------|--------|-------------|
| **RDD2022 India** (Figshare) | potholes + drainage | CC BY-SA 4.0 | ~9,665 (India) | Dashcam/phone, Delhi roads |
| **TACO** (GitHub) | others | CC BY 4.0 | ~1,500 | Phone photos, trash in wild |
| **QR4Change** (Mendeley) | potholes + others | CC BY 4.0 | ~5,000 | Phone photos, Pune civic issues |
| **Kaggle: Flood datasets** | drainage | Various | ~9,000+ | Urban flooding |
| **Kaggle: Urban Issues** | others | Various | Multi-class | Fallen trees, vandalism, garbage |
| **Kaggle: Garbage Classification** | others | CC | 15,000+ | In-the-wild waste |

**After this cell, expected counts**: drainage 800+, others 800+, potholes 2000+, streetlight 1700+

### Setup Kaggle API (optional but recommended)
If you want the Kaggle datasets (flood, urban issues, garbage):
1. Go to kaggle.com → Account → Create API Token → download `kaggle.json`
2. Upload it to Colab: `from google.colab import files; files.upload()` then `!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json`

In [ ]:
"""
Premium dataset acquisition — CONDITIONAL.
If my_dataset.zip was pre-built locally with sufficient images (800+ per category),
this cell SKIPS all downloads. Only supplements categories below threshold.
"""
import os, glob, shutil, random, zipfile
from PIL import Image

random.seed(42)
os.chdir('/content/work')

def ensure_dir(d):
    os.makedirs(d, exist_ok=True)
    return d

def count_cat(c):
    return len(glob.glob(f'my_dataset/{c}/*'))

def copy_images(src_list, dst_dir, prefix, max_count, quality=90, min_size=50):
    """Copy and validate images to a category folder."""
    ensure_dir(dst_dir)
    random.shuffle(src_list)
    copied = 0
    for img_path in src_list:
        if copied >= max_count:
            break
        try:
            img = Image.open(img_path).convert('RGB')
            w, h = img.size
            if min(w, h) < min_size:
                continue
            if max(w, h) > 1024:
                ratio = 1024 / max(w, h)
                img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
            dst = os.path.join(dst_dir, f'{prefix}_{copied:04d}.jpg')
            img.save(dst, 'JPEG', quality=quality)
            copied += 1
        except Exception:
            pass
    return copied

# ============================================================
# GATE: Skip if all categories above threshold
# ============================================================
MIN_THRESHOLD = 800
counts = {c: count_cat(c) for c in ['drainage', 'others', 'potholes', 'streetlight']}
print("=== Dataset counts after cleanup ===")
for c, n in counts.items():
    status = "✓ sufficient" if n >= MIN_THRESHOLD else "✗ needs supplement"
    print(f"  {c}: {n} ({status})")

if all(n >= MIN_THRESHOLD for n in counts.values()):
    print(f"\n✓ ALL categories above {MIN_THRESHOLD}. SKIPPING all downloads.")
    print("  Dataset was pre-built locally — no Colab downloads needed.")
else:
    print(f"\n⚠ Some categories below {MIN_THRESHOLD}. Attempting downloads...")
    # (fallback download logic — only runs if local pre-build was incomplete)
    need_drainage = counts['drainage'] < MIN_THRESHOLD
    need_others = counts['others'] < MIN_THRESHOLD
    need_potholes = counts['potholes'] < MIN_THRESHOLD

    if need_others:
        print("\n[1] TACO — trash/litter for 'others'")
        TACO_URL = "https://github.com/pedropro/TACO/archive/refs/heads/master.zip"
        taco_zip = '/content/taco.zip'
        if not os.path.exists(taco_zip):
            !wget -q --show-progress -O "$taco_zip" "$TACO_URL" 2>&1 || true
        if os.path.exists(taco_zip) and os.path.getsize(taco_zip) > 100000:
            ensure_dir('/content/taco')
            !unzip -q -o "$taco_zip" -d /content/taco 2>/dev/null || true
            taco_imgs = []
            for root, dirs, files in os.walk('/content/taco'):
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        fpath = os.path.join(root, f)
                        if os.path.getsize(fpath) > 5000:
                            taco_imgs.append(fpath)
            n = copy_images(taco_imgs, 'my_dataset/others', 'taco', 600)
            print(f"  ✓ Others: +{n} TACO images")

    if need_potholes:
        print("\n[2] RDD2022 India — road damage for 'potholes'")
        RDD_URL = "https://figshare.com/ndownloader/articles/21431547/versions/2"
        rdd_zip = '/content/rdd2022.zip'
        if not os.path.exists(rdd_zip):
            !wget -q --show-progress -O "$rdd_zip" "$RDD_URL" 2>&1 || true
        if os.path.exists(rdd_zip) and os.path.getsize(rdd_zip) > 1000000:
            ensure_dir('/content/rdd2022')
            !unzip -q -o "$rdd_zip" -d /content/rdd2022 2>/dev/null || true
            rdd_imgs = []
            for root, dirs, files in os.walk('/content/rdd2022'):
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        fpath = os.path.join(root, f)
                        if os.path.getsize(fpath) > 5000:
                            rdd_imgs.append(fpath)
            n = copy_images(rdd_imgs, 'my_dataset/potholes', 'rdd22', 500)
            print(f"  ✓ Potholes: +{n} RDD2022 images")

    if need_drainage:
        print("\n[3] Flood dataset supplement for 'drainage'")
        print("  ⚠ Kaggle API needed — configure ~/.kaggle/kaggle.json or add images manually")

    print("\n=== After downloads ===")
    for c in ['drainage', 'others', 'potholes', 'streetlight']:
        print(f"  {c}: {count_cat(c)}")

## 4d. JPEG Compression Augmentation (Domain Gap Mitigation)

Citizens upload photos through the frontend which compresses them to **800×800 JPEG at 70% quality** via canvas. Training images are typically higher quality originals. This cell re-saves 30% of training images at lower JPEG quality (50–80%) to simulate the double-compression pipeline and close the domain gap.

In [ ]:
import os, glob, random
from PIL import Image

random.seed(42)
os.chdir('/content/work')
resaved = 0
CLASSES = ['drainage', 'others', 'potholes', 'streetlight']

for c in CLASSES:
    d = f'my_dataset/{c}'
    if not os.path.isdir(d):
        continue
    files = [f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    targets = random.sample(files, min(len(files), int(len(files) * 0.3)))
    for f in targets:
        fpath = os.path.join(d, f)
        try:
            img = Image.open(fpath).convert('RGB')
            quality = random.randint(50, 80)
            out = os.path.splitext(fpath)[0] + '.jpg'
            img.save(out, 'JPEG', quality=quality)
            if fpath != out and os.path.exists(fpath):
                os.remove(fpath)
            resaved += 1
        except Exception as e:
            print(f'  Skip {f}: {e}')

print(f'Re-saved {resaved} images with JPEG compression artifacts (quality 50-80%)')
print('This simulates the frontend compression pipeline to close the domain gap.')

print('\n=== READY FOR TRAINING ===')
for c in CLASSES:
    n = len(glob.glob(f'my_dataset/{c}/*'))
    print(f'  {c}: {n} images')

## 5. Train (GPU)
3-stage EfficientNetV2S progressive transfer learning:
- **Stage 1**: Head training (backbone frozen), 30 epochs, LR=1e-3
- **Stage 2**: Unfreeze top 60 layers, cosine LR from 3e-5, 25 epochs, CutMix/Mixup
- **Stage 3**: Full fine-tune, LR=5e-6, 15 epochs

On a T4: **15–30 min** total. On A100: **5–10 min**.
`--batch 64` uses GPU better. Drop to 32 if OOM.

In [ ]:
!python train_civic_model.py --batch 64
# GATE: if it prints [ABORT] collapse guard and exits, the model is bad — do NOT ship it.
# Success ends with '=== Training Complete ===' and writes civic_model_tfjs/.

## 5b. Temperature Calibration (run after training succeeds)
Fits temperature scaling on the validation set, computes per-class thresholds, writes `civic_thresholds.json`.

In [ ]:
# Copy temperature_scaling.py from Drive (if available) and run calibration
import shutil, os
temp_script = os.path.join(SRC, 'temperature_scaling.py')
if os.path.exists(temp_script):
    shutil.copy(temp_script, 'temperature_scaling.py')
    !python temperature_scaling.py --target-recall 0.92
    print('\nCalibration complete. civic_thresholds.json written.')
else:
    print('temperature_scaling.py not found in Drive/FixMyCity — run calibration locally after download.')

## 6. Package artifacts + download

In [ ]:
import os, shutil, zipfile
from google.colab import files

os.chdir('/content/work')

# Package all training artifacts into a single zip
artifacts = 'civic_artifacts'
if os.path.isdir(artifacts):
    shutil.rmtree(artifacts)
os.makedirs(artifacts)

# Copy TFJS model
if os.path.isdir('civic_model_tfjs'):
    shutil.copytree('civic_model_tfjs', f'{artifacts}/civic_model_tfjs')
    print("✓ civic_model_tfjs/ included")

# Copy Keras model
if os.path.isfile('civic_model.keras'):
    shutil.copy('civic_model.keras', f'{artifacts}/civic_model.keras')
    print("✓ civic_model.keras included")

# Copy thresholds
if os.path.isfile('civic_thresholds.json'):
    shutil.copy('civic_thresholds.json', f'{artifacts}/civic_thresholds.json')
    print("✓ civic_thresholds.json included")

# Copy labels
if os.path.isfile('civic_labels.json'):
    shutil.copy('civic_labels.json', f'{artifacts}/civic_labels.json')
    print("✓ civic_labels.json included")

# Create zip
zip_path = 'civic_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(artifacts):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, '.')
            zf.write(fpath, arcname)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\n✓ civic_artifacts.zip created ({size_mb:.1f} MB)")

# Also save to Drive for persistence
drive_dst = os.path.join(SRC, 'civic_artifacts.zip')
shutil.copy(zip_path, drive_dst)
print(f"✓ Saved to Drive: {drive_dst}")

# Trigger browser download
files.download(zip_path)

## Back on your local machine
```bash
cd backend
# unzip civic_artifacts.zip here, overwriting civic_model.keras + civic_model_tfjs/
python temperature_scaling.py --target-recall 0.92
python audit_dataset.py
python build_others_exemplars.py --extra internet_images
npm run dev   # then check /api/health
```